<a href="https://colab.research.google.com/github/matias-ferrero/algo2-stats/blob/main/stats_algo2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# STATS

## Dependencias

In [ ]:
# Imports para conectividad y autenticación con Google Sheets
import gspread as gs # API de Python para manipular Google Sheets
from google.colab import auth # Maneja la autenticación del usuario en entornos de Colab
from google.auth import default # Obtiene las credenciales por defecto para acceder a la API
from google.auth.exceptions import GoogleAuthError # Maneja errores durante el proceso de autenticación

# Imports para el logger
import logging
import sys

# Imports para manipulación y análisis de datos
import pandas as pd
import unicodedata

# Imports para visualización de datos y gráficos/figuras
import matplotlib.pyplot as plt

## Constantes

In [ ]:
URL = 'https://docs.google.com/spreadsheets/d/1CTtpW6oQ0KYsLwz0sL75q_-51-sdZIXGb1wl09WfOng/edit?gid=0#gid=0'

In [ ]:
COLUMNS = ['Padron', 'intentos', 'Hora entrega', 'status', 'Corrector', 'Aprobado', 'Codigo', 'Pruebas', 'Informe', 'Interaccion', 'Nota final']

In [ ]:
TP0_SHEET = "TP0"
TP1_SHEET = "TP1"
LISTA_SHEET = "LISTA"
ABB_SHEET = "ABB"
HASH_SHEET = "HASH"
TP2_SHEET = "TP2"

# La planilla de 2026+ renombró la hoja HASH a DIC; ambas se refieren al mismo TP
SHEET_ALIASES = {"DIC": HASH_SHEET}

# Solo se procesan estas hojas; cualquier otra hoja (Karma, Parametros, Final, Asignaciones, ...) se ignora
VALID_SHEETS = {TP0_SHEET, TP1_SHEET, LISTA_SHEET, ABB_SHEET, HASH_SHEET, TP2_SHEET}

## Logger

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format='%(levelname)s: %(message)s',
    force=True # Asegura que la configuración se aplique en el entorno de Colab
)
logger = logging.getLogger(__name__)

## Preprocesamiento

### Autenticación

Establece una conexión entre `Google Colab` y la API de `Google Sheets`. Este proceso requiere ***verificación de identidad del usuario*** e inicializa un cliente para gestionar las planillas.

**Pasos para autenticarse:**

1. Iniciá el login ejecutando esta sección.
2. Va a aparecer una ventana emergente; permití el proceso de login.
3. Va a aparecer una ventana de inicio de sesión de Google; seleccioná una cuenta que tenga acceso a las planillas.
4. Otorgá los permisos para autorizar a Google Cloud SDK a ver y gestionar tus archivos de Google Drive y Google Sheets.
5. Verificación: una vez completado el login, la consola va a mostrar "Authentication successful", y vas a poder correr el resto del script.

In [ ]:
try:
    logger.info("Starting authentication process...")

    # Abre el prompt de OAuth2 de Google para el usuario
    auth.authenticate_user()

    # Obtiene las credenciales del entorno y autoriza al cliente de gspread
    credential, _ = default()
    client = gs.authorize(credential)

    logger.info("Authentication successful: Client initialized.")

except GoogleAuthError as auth_err:
    # Maneja problemas específicamente relacionados al flujo de login o a los permisos
    logger.error(f"Authentication Error: Please ensure you complete the login flow.\nDetails: {auth_err}")
    raise SystemExit("Execution stopped: Unauthorized access.") from None

except Exception as e:
    # Problemas de red o fallas inesperadas del sistema
    logger.critical(f"Unexpected Error during connection: {e}")
    raise SystemExit("Execution stopped due to a critical error.") from None

### Normalización

Distintos cuatrimestres nombraron las mismas columnas de manera diferente (por ejemplo, `Nota final` vs `Nota Final`, `intentos` vs `Intentos`, `Padron` vs `Columna 1`/` gv`, o `Hora entrega` dividida en `Date` (UTC) + `Hora` (UTC-3)). Esta sección mapea cada variante conocida al nombre canónico usado en `COLUMNS` —usando siempre `Hora`, que está en horario local (UTC-3) igual que la `Hora entrega` de 2025, y no `Date`, que viene en UTC— para que el resto de la notebook no necesite preocuparse por qué versión de la planilla está leyendo.

In [ ]:
def _normalize_key(value):
    """Forma en minúsculas, sin acentos y sin espacios sobrantes, usada para matchear nombres de columna."""
    ascii_value = unicodedata.normalize('NFKD', str(value)).encode('ascii', 'ignore').decode('ascii')
    return ascii_value.strip().lower()

# Columnas que cambiaron de nombre entre cuatrimestres pero representan el mismo campo
COLUMN_ALIASES = {
    'gv': 'Padron',          # La hoja LISTA (2026+) usa " gv" en vez de "Padron"
    'columna 1': 'Padron',   # La hoja HASH (2025) usa "Columna 1" en vez de "Padron"
    'hora': 'Hora entrega',  # 2026+ dividió "Hora entrega" en "Date" (UTC) + "Hora" (UTC-3, la que se usaba antes)
}

_CANONICAL_BY_KEY = {_normalize_key(col): col for col in COLUMNS}
_CANONICAL_BY_KEY.update(COLUMN_ALIASES)

def normalize_columns(df):
    """Renombra columnas para que tanto el formato de 2025 como el de 2026+ coincidan con COLUMNS."""
    rename = {
        col: _CANONICAL_BY_KEY[_normalize_key(col)]
        for col in df.columns
        if _normalize_key(col) in _CANONICAL_BY_KEY
    }
    return df.rename(columns=rename)

### Hojas

Transforma todas las hojas disponibles en DataFrames para su posterior procesamiento y análisis.

- Ignora las hojas que no están en `VALID_SHEETS` (aplicando antes los alias de `SHEET_ALIASES`, por ejemplo `DIC` → `HASH`).
- Recorre cada hoja restante del archivo, usando la primera fila como encabezado y el resto como datos.
- Valida el contenido de cada hoja, distinguiendo entre datos activos, solo encabezados, o pestañas vacías.
- Detiene la ejecución si no se recolectaron datos (todas las hojas son inválidas).

In [ ]:
sh = client.open_by_url(URL)
csvs = sh.worksheets()

all_sheets = {}

for csv in csvs:
    title = SHEET_ALIASES.get(csv.title, csv.title)

    if title not in VALID_SHEETS:
        logger.debug(f"Sheet '{csv.title}' is not one of the tracked sheets. Skipping.")
        continue

    try:
        data = csv.get_all_values()

        if len(data) > 1:
            df = normalize_columns(pd.DataFrame(data[1:], columns=data[0]))
            all_sheets[title] = df

            logger.info(f"Sheet '{csv.title}' successfully converted to DataFrame.")

        elif len(data) == 1:
            logger.warning(f"Sheet '{csv.title}' contains only headers and no data.")
        else:
            logger.debug(f"Sheet '{csv.title}' is completely empty.")

    except Exception as e:
        logger.error(f"Failed to process sheet '{csv.title}'.\nDetails: {e}")

if not all_sheets:
    logger.error(f"Failed to process every sheet")
    raise SystemExit("Execution stopped: No sheets were processed.")

### Columnas

Filtra cada columna necesaria para el análisis, y solo conserva las hojas que contengan al menos una columna coincidente.

In [ ]:
sheets = {}

for title, df in all_sheets.items():
    columns = [col for col in COLUMNS if col in df.columns]

    if columns:
        sheets[title] = df[columns].copy()
        columns_str = ", ".join(columns)
        logger.info(f"'{title}': Retained columns: [{columns_str}]")
    else:
        logger.warning(f"'{title}': No matching columns found. Skipping.")

### Vista previa

Imprime el comienzo de cada hoja después de filtrar las columnas.

In [ ]:
for title, df in sheets.items():
    print(f"Mostrando datos de la hoja: {title}")
    display(df.head(1))

## Estadísticas

### Utilidades

Esta función actúa como un formateador personalizado para las etiquetas de datos de los gráficos de torta usados en esta sección. Toma el porcentaje relativo y el tamaño total de la muestra para calcular la cantidad exacta de estudiantes por porción de cada gráfico.

También suprime las etiquetas de 0% para mejorar la visualización de los gráficos, manteniéndolos legibles.

In [ ]:
def fmt_labels(pct, total):
    if pct == 0:
        return ""

    absolute = int(round(pct/100.*total))
    return f"{pct:.1f}%\n({absolute})"

### TP0

#### N/A TP1

Este módulo identifica a los estudiantes que entregaron el TP0 pero nunca llegaron a entregar el TP1, es decir, bajas que ocurrieron entre ambos trabajos prácticos. Cruzando esos padrones con su resultado en el TP0, podemos estimar en qué proporción esas bajas se dieron después de aprobar el TP0 (posible abandono de la materia con un TP ya aprobado) o después de desaprobarlo (posible abandono tras la primera dificultad).

- Calcula la diferencia entre los padrones presentes en `TP0` y los presentes en `TP1`, para aislar a los estudiantes ausentes en TP1.
- Cruza esos padrones con su resultado (`Aprobado`) en TP0, clasificándolos en **Aprobado**, **Desaprobado**, o **N/A** (si el resultado no está disponible), usando el mismo esquema de categorías que la sección de Desaprobados TP1.
- Si todos los estudiantes que entregaron TP0 también entregaron TP1, el módulo lo informa por consola y no genera ningún gráfico.

Como resultado, el módulo devuelve un DataFrame con los datos de esos estudiantes, más un gráfico de torta para visualizar la proporción de cada resultado.

##### Implementación

Esta función gestiona la generación y visualización completa del gráfico de torta de este módulo. Reutiliza el mismo formateador de etiquetas (`fmt_labels`) y el mismo esquema de colores y categorías que el gráfico de la sección de Desaprobados TP1, para que ambas visualizaciones sean consistentes entre sí.

In [ ]:
def missing_tp1_pie_chart(categories, categories_count, students_count):
  plt.figure(figsize=(10, 5))

  colors = ['#66ff99', '#ff3333', '#3366ff']
  font_style = {'family': 'serif', 'style': 'italic'}

  patches, texts, autotexts = plt.pie(
      categories_count,
      labels=None,
      autopct=lambda p: fmt_labels(p, students_count),
      colors=colors,
      shadow={'ox': 0.03, 'shade': 0.2},
      pctdistance=0.8,
      explode=[0.05] * len(categories)
  )

  plt.setp(autotexts, size=11, weight="bold", color="black", family=font_style['family'])

  legend_labels = [
      f"{cat}: {val} ({(val/students_count)*100:.1f}%)"
      if students_count > 0 else f"{cat}: 0 (0.0%)"
      for cat, val in categories_count.items()
  ]

  plt.legend(
      patches,
      legend_labels,
      title=f"Referencias (Total: {students_count} alumnos)",
      title_fontproperties={**font_style, 'size': 13, 'weight': 'bold'},
      prop={**font_style, 'size': 12},
      loc="center left",
      bbox_to_anchor=(1.05, 0.5),
      labelspacing=1.5,
      handletextpad=1.2,
      borderpad=1.5
  )

  plt.suptitle(
      'Ausentes en TP1 según su TP0',
      x=0.5, y=0.92,
      fontproperties={**font_style, 'size': 20, 'weight': 'bold'},
      ha='center'
  )

  plt.figtext(
      0.5, 0.10, "Figura 1.1",
      ha="center", fontsize=11, family='serif', weight='bold', color='#333333'
  )

  plt.figtext(
      0.5, 0.04,
      "Cátedra Mendez/Pandolfo - Algoritmos y Estructuras de Datos - FIUBA",
      ha="center", fontsize=9, family='serif', style='italic', color='#555555',
      bbox={"facecolor":"orange", "alpha":0.1, "pad":5, "edgecolor":"none"}
  )

  plt.subplots_adjust(top=0.85, bottom=0.22, right=0.5)

  plt.tight_layout()
  plt.show()

Esta sección calcula la diferencia entre los padrones de TP0 y TP1 para aislar a los estudiantes ausentes en TP1, y categoriza su resultado en TP0. Tanto `Padron` como `Aprobado` son columnas comunes a ambos esquemas de planilla (2025 y 2026+): `Padron` ya queda normalizada por `normalize_columns` (a partir de `gv`/`Columna 1` según corresponda), y `Aprobado` no cambió de nombre entre cuatrimestres, por lo que esta implementación funciona sin cambios contra cualquiera de los dos formatos.

In [ ]:
missing_tp1_categories = ['TP0 Aprobado', 'TP0 Desaprobado', 'TP0 N/A']

padrones_en_tp0 = set(sheets[TP0_SHEET]['Padron'])
padrones_en_tp1 = set(sheets[TP1_SHEET]['Padron'])
padrones_ausentes = padrones_en_tp0 - padrones_en_tp1

# Estudiantes que entregaron TP0 pero no TP1
missing_tp1 = sheets[TP0_SHEET][sheets[TP0_SHEET]['Padron'].isin(padrones_ausentes)][['Padron', 'Aprobado']].copy()

# Mapea la columna "Aprobado" a las categorías
missing_tp1['TP0_Final'] = missing_tp1['Aprobado'].map({'Si': 'TP0 Aprobado', 'No': 'TP0 Desaprobado'}).fillna('TP0 N/A')
missing_tp1.drop(columns=['Aprobado'], inplace=True)

# Cuenta las ocurrencias por categoría y reindexa para asegurar que existan las 3 categorías (incluso si el conteo es 0)
missing_tp1_categories_count = missing_tp1['TP0_Final'].value_counts().reindex(missing_tp1_categories, fill_value=0)

# Calcula el total de estudiantes
missing_tp1_students_count = missing_tp1_categories_count.sum()

##### Resultado

In [ ]:
if missing_tp1_students_count > 0:
    display(missing_tp1)
    print(f"\n")
    missing_tp1_pie_chart(missing_tp1_categories, missing_tp1_categories_count, missing_tp1_students_count)
else:
    logger.info("Everyone that finished TP0 also finished TP1.")

### TP1

#### APROBADOS TP1

In [ ]:
# TODO

#### DESAPROBADOS TP1

Este módulo analiza el desempeño previo (opcional) en el TP0 de quienes desaprobaron el TP1. Esto puede darnos una idea de por qué tantos estudiantes fallan en esta etapa, dado que lo único que tienen antes es el TP0 (aparte de RPL).

- Aísla a la población de estudiantes que desaprobó el TP1 y, a partir de esos registros, cruza sus acciones en el TP0 para determinar si aprobaron, desaprobaron, o nunca entregaron el trabajo anterior.
- Identifica a los estudiantes que están presentes en el TP1 pero ausentes en los registros del TP0, y los clasifica como *"No disponible"* (N/A), para también tenerlos en cuenta.
- También asegura que el análisis siempre considere los tres resultados posibles del TP0 (**Aprobado, Desaprobado, N/A**), incluso si alguno de esos grupos tiene cero estudiantes en un cuatrimestre en particular.

Como resultado, el módulo devuelve un DataFrame con los datos importantes de los estudiantes, más un gráfico de torta para una mejor visualización de los números.

##### Implementación

Esta función gestiona la generación y visualización completa del gráfico de torta de este módulo. Encapsula todas las configuraciones estéticas y estructurales necesarias para transformar los datos procesados en la figura que se muestra a continuación.

In [ ]:
def failed_tp1_pie_chart(categories, categories_count, students_count):
  plt.figure(figsize=(10, 5))

  colors = ['#66ff99', '#ff3333', '#3366ff']
  font_style = {'family': 'serif', 'style': 'italic'}

  patches, texts, autotexts = plt.pie(
      categories_count,
      labels=None,
      autopct=lambda p: fmt_labels(p, students_count),
      colors=colors,
      shadow={'ox': 0.03, 'shade': 0.2},
      pctdistance=0.8,
      explode=[0.05] * len(categories)
  )

  plt.setp(autotexts, size=11, weight="bold", color="black", family=font_style['family'])

  legend_labels = [
      f"{cat}: {val} ({(val/students_count)*100:.1f}%)"
      if students_count > 0 else f"{cat}: 0 (0.0%)"
      for cat, val in categories_count.items()
  ]

  plt.legend(
      patches,
      legend_labels,
      title=f"Referencias (Total: {students_count} alumnos)",
      title_fontproperties={**font_style, 'size': 13, 'weight': 'bold'},
      prop={**font_style, 'size': 12},
      loc="center left",
      bbox_to_anchor=(1.05, 0.5),
      labelspacing=1.5,
      handletextpad=1.2,
      borderpad=1.5
  )

  plt.suptitle(
      'Desaprobados en TP1 según su TP0',
      x=0.5, y=0.92,
      fontproperties={**font_style, 'size': 20, 'weight': 'bold'},
      ha='center'
  )

  plt.figtext(
      0.5, 0.10, "Figura 1.2",
      ha="center", fontsize=11, family='serif', weight='bold', color='#333333'
  )

  plt.figtext(
      0.5, 0.04,
      "Cátedra Mendez/Pandolfo - Algoritmos y Estructuras de Datos - FIUBA",
      ha="center", fontsize=9, family='serif', style='italic', color='#555555',
      bbox={"facecolor":"orange", "alpha":0.1, "pad":5, "edgecolor":"none"}
  )

  plt.subplots_adjust(top=0.85, bottom=0.22, right=0.5)

  plt.tight_layout()
  plt.show()

Esta sección filtra a los estudiantes que desaprobaron y categoriza esos registros.

In [ ]:
categories = ['TP0 Aprobado', 'TP0 Desaprobado', 'TP0 N/A']

# Filtra a los estudiantes que desaprobaron el TP1
failed_tp1 = sheets[TP1_SHEET][sheets[TP1_SHEET]['Nota final'] == 'Desaprobado'].copy()

# Cruza con los datos de TP0 usando "Padron" y conservando "Aprobado"
failed_tp1 = failed_tp1.merge(
    sheets[TP0_SHEET][['Padron', 'Aprobado']],
    on='Padron',
    how='left'
)

# Mapea la columna "Aprobado" a las categorías
failed_tp1['TP0_Final'] = failed_tp1['Aprobado'].map({'Si': 'TP0 Aprobado', 'No': 'TP0 Desaprobado'}).fillna('TP0 N/A')
failed_tp1.drop(columns=['Aprobado'], inplace=True)

# Cuenta las ocurrencias por categoría y reindexa para asegurar que existan las 3 categorías (incluso si el conteo es 0)
categories_count = failed_tp1['TP0_Final'].value_counts().reindex(categories, fill_value=0)

# Calcula el total de estudiantes
students_count = categories_count.sum()

##### Resultado

In [ ]:
display(failed_tp1)
print(f"\n")
failed_tp1_pie_chart(categories, categories_count, students_count)